In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import geopandas as gpd
import os

In [2]:
filePath = "chile_ebd_relSep-2026.parquet"
df = pd.read_parquet(filePath)
print("Data Fully Loaded!")
display(df.head().T)

Data Fully Loaded!


,0,1,2,3,4
GLOBAL UNIQUE IDENTIFIER,URN:CornellLabOfOrnithology:EBIRD:OBS162674606,URN:CornellLabOfOrnithology:EBIRD:OBS975592694,URN:CornellLabOfOrnithology:EBIRD:OBS175214333,URN:CornellLabOfOrnithology:EBIRD:OBS139313918,URN:CornellLabOfOrnithology:EBIRD:OBS779288312
LAST EDITED DATE,2025-10-12 19:47:59.111233,2025-10-12 19:45:58.942104,2023-03-27 14:59:09.858277,2025-03-20 08:40:40.678111,2026-06-03 12:38:16.985757
TAXONOMIC ORDER,524,6533,5896,6494,33931
CATEGORY,species,species,species,species,species
TAXON CONCEPT ID,avibase-0B8AF41F,avibase-E6536E4E,avibase-AE459A9B,avibase-57C93A74,avibase-11755EFA
COMMON NAME,Chiloe Wigeon,Kelp Gull,Southern Lapwing,Franklin's Gull,Yellow-winged Blackbird
SCIENTIFIC NAME,Mareca sibilatrix,Larus dominicanus,Vanellus chilensis,Leucophaeus pipixcan,Agelasticus thilius
SUBSPECIES COMMON NAME,None,None,None,None,None
SUBSPECIES SCIENTIFIC NAME,None,None,None,None,None
EXOTIC CODE,None,None,None,None,None


In [3]:
rows, cols = df.shape
colNames = list(df.columns)
print("Rows:", rows)
print("Columns:", cols)
print("Column Names:")
for c in colNames:
    isUnhashable = df[c].apply(lambda x: isinstance(x, (list, dict, set))).any()
    workingSeries = df[c].apply(lambda x: str(x) if isinstance(x, (list, dict, set)) else x) if isUnhashable else df[c]
    dtype = df[c].dtype
    uniqueCount = workingSeries.nunique()
    if str(dtype) == 'object':
        cleanSeries = workingSeries.dropna()
        maxLen = max((len(str(val)) for val in cleanSeries), default = 0)
        detail = f"Max Length: {maxLen}"
    else:
        detail = f"Range: {df[c].min()}, {df[c].max()}"
    print(f"\t- {c}:\n\t\t- Type: {dtype}\n\t\t- Unique Values: {uniqueCount}\n\t\t- {detail}")

Rows: 10311822
Columns: 52
Column Names:
	- GLOBAL UNIQUE IDENTIFIER:
		- Type: object
		- Unique Values: 10311822
		- Max Length: 47
	- LAST EDITED DATE:
		- Type: object
		- Unique Values: 604791
		- Max Length: 26
	- TAXONOMIC ORDER:
		- Type: int64
		- Unique Values: 949
		- Range: 38, 35853
	- CATEGORY:
		- Type: object
		- Unique Values: 8
		- Max Length: 10
	- TAXON CONCEPT ID:
		- Type: object
		- Unique Values: 949
		- Max Length: 16
	- COMMON NAME:
		- Type: object
		- Unique Values: 778
		- Max Length: 46
	- SCIENTIFIC NAME:
		- Type: object
		- Unique Values: 778
		- Max Length: 47
	- SUBSPECIES COMMON NAME:
		- Type: object
		- Unique Values: 172
		- Max Length: 48
	- SUBSPECIES SCIENTIFIC NAME:
		- Type: object
		- Unique Values: 172
		- Max Length: 48
	- EXOTIC CODE:
		- Type: object
		- Unique Values: 3
		- Max Length: 1
	- OBSERVATION COUNT:
		- Type: object
		- Unique Values: 1828
		- Max Length: 7
	- BREEDING CODE:
		- Type: object
		- Unique Values: 23
		- Max Lengt

In [4]:
colNamesForMatrix = [ c for c in colNames if c not in ["GLOBAL UNIQUE IDENTIFIER", "LAST EDITED DATE", "OBSERVATION DATE", "TIME OBSERVATIONS STARTED"] ]
rsMatrix = pd.DataFrame(-1, index = colNamesForMatrix, columns = colNamesForMatrix)
for i, col1 in enumerate(colNamesForMatrix):
    if df[col1].nunique() <= 1:
        continue
    print(f"- {col1}:")
    for col2 in colNamesForMatrix[i + 1:]:
        if df[col2].nunique() <= 1:
            continue
        oneToN = df.groupby(col1)[col2].nunique().max() <= 1
        nToOne = df.groupby(col2)[col1].nunique().max() <= 1
        if oneToN and nToOne:
            values = ["1:1"] * 2
        elif oneToN:
            values = ["1:n", "n:1"]
        elif nToOne:
            values = ["n:1", "1:n"]
        else:
            values = ["n:n"] * 2
        if oneToN or nToOne:
            print(f"\t- {col1} - {col2}: {values[0]}")
            print(f"\t- {col2} - {col1}: {values[1]}")
        rsMatrix.loc[col1, col2] = values[0]
        rsMatrix.loc[col2, col1] = values[1]

- TAXONOMIC ORDER:
	- TAXONOMIC ORDER - CATEGORY: 1:n
	- CATEGORY - TAXONOMIC ORDER: n:1
	- TAXONOMIC ORDER - TAXON CONCEPT ID: 1:1
	- TAXON CONCEPT ID - TAXONOMIC ORDER: 1:1
	- TAXONOMIC ORDER - COMMON NAME: 1:n
	- COMMON NAME - TAXONOMIC ORDER: n:1
	- TAXONOMIC ORDER - SCIENTIFIC NAME: 1:n
	- SCIENTIFIC NAME - TAXONOMIC ORDER: n:1
	- TAXONOMIC ORDER - SUBSPECIES COMMON NAME: 1:1
	- SUBSPECIES COMMON NAME - TAXONOMIC ORDER: 1:1
	- TAXONOMIC ORDER - SUBSPECIES SCIENTIFIC NAME: 1:1
	- SUBSPECIES SCIENTIFIC NAME - TAXONOMIC ORDER: 1:1
- CATEGORY:
	- CATEGORY - TAXON CONCEPT ID: n:1
	- TAXON CONCEPT ID - CATEGORY: 1:n
	- CATEGORY - SUBSPECIES COMMON NAME: n:1
	- SUBSPECIES COMMON NAME - CATEGORY: 1:n
	- CATEGORY - SUBSPECIES SCIENTIFIC NAME: n:1
	- SUBSPECIES SCIENTIFIC NAME - CATEGORY: 1:n
- TAXON CONCEPT ID:
	- TAXON CONCEPT ID - COMMON NAME: 1:n
	- COMMON NAME - TAXON CONCEPT ID: n:1
	- TAXON CONCEPT ID - SCIENTIFIC NAME: 1:n
	- SCIENTIFIC NAME - TAXON CONCEPT ID: n:1
	- TAXON CONCEPT 

In [ ]:
def parseDemographics(text):
    if pd.isna(text):
        return []
    pattern = r"([^,]+),\s*([^\(]+)\s*\((\d+)\)"
    result = []
    for item in text.split("; "):
        match = re.search(pattern, item.strip())
        if match:
            result.append(
                {
                    "SEX": match.group(1).strip(),
                    "AGE": match.group(2).strip(),
                    "QUANTITY": int(match.group(3)),
                }
            )
    return result

os.makedirs("cleaned_ebird_data", exist_ok = True)
chunkSize = 500000
for i, start in enumerate(range(0, len(df), chunkSize)):
    chunk = df.iloc[start:start + chunkSize].copy()
    chunk = chunk.drop(columns=["BCR CODE", "USFWS CODE", "ATLAS BLOCK", "OBSERVATION DATE", "REASON"], errors = "ignore")
    chunk["LAST EDITED DATE"] = pd.to_datetime(chunk["LAST EDITED DATE"])
    chunk["OBSERVATION COUNT"] = pd.to_numeric(chunk["OBSERVATION COUNT"], errors = "coerce")
    chunk["TIME OBSERVATIONS STARTED"] = pd.to_datetime(df["OBSERVATION DATE"].iloc[start:start + chunkSize].str.cat(chunk["TIME OBSERVATIONS STARTED"], sep = " "))
    chunk["AGE/SEX"] = chunk["AGE/SEX"].apply(parseDemographics)
    chunk.to_parquet(f"cleaned_ebird_data/chunk_{i}.parquet", index = False)
    print(f"Processed and saved chunk {i + 1}")
print("All chunks processed and saved to disk.")

In [5]:
df1 = pd.read_parquet("cleaned_ebird_data/")
matchingIndexes = []
for idx, val in df1["AGE/SEX"].items():
    if isinstance(val, (list, np.ndarray)) and len(val) > 0:
        matchingIndexes.append(idx)
    if len(matchingIndexes) >= 5:
        break
display(df1.loc[matchingIndexes].T)

,9,36,87,221,230
GLOBAL UNIQUE IDENTIFIER,URN:CornellLabOfOrnithology:EBIRD:OBS164354504,URN:CornellLabOfOrnithology:EBIRD:OBS170360876,URN:CornellLabOfOrnithology:EBIRD:OBS168674839,URN:CornellLabOfOrnithology:EBIRD:OBS169658365,URN:CornellLabOfOrnithology:EBIRD:OBS170956509
LAST EDITED DATE,2025-10-12 19:49:38.348502,2023-04-14 16:07:06.040350,2024-10-14 10:34:01.582101,2025-10-12 19:47:59.514851,2023-03-27 14:51:29.809404
TAXONOMIC ORDER,17029,5936,7487,7431,6489
CATEGORY,species,species,species,species,species
TAXON CONCEPT ID,avibase-1456993D,avibase-A931ED99,avibase-7DE15211,avibase-56036504,avibase-683C03D9
COMMON NAME,Vermilion Flycatcher,Two-banded Plover,Black-faced Ibis,Neotropic Cormorant,Dolphin Gull
SCIENTIFIC NAME,Pyrocephalus rubinus,Anarhynchus falklandicus,Theristicus melanopis,Nannopterum brasilianum,Leucophaeus scoresbii
SUBSPECIES COMMON NAME,None,None,None,None,None
SUBSPECIES SCIENTIFIC NAME,None,None,None,None,None
EXOTIC CODE,None,None,None,None,None


In [6]:
print("Column Names:")
for c in list(df1.columns):
    if c == "AGE/SEX":
        continue
    isUnhashable = df1[c].apply(lambda x: isinstance(x, (list, dict, set))).any()
    workingSeries = df1[c].apply(lambda x: str(x) if isinstance(x, (list, dict, set)) else x) if isUnhashable else df1[c]
    dtype = df1[c].dtype
    uniqueCount = workingSeries.nunique()
    if str(dtype) == 'object':
        cleanSeries = workingSeries.dropna()
        maxLen = max((len(str(val)) for val in cleanSeries), default = 0)
        detail = f"Max Length: {maxLen}"
    else:
        detail = f"Range: {df1[c].min()}, {df1[c].max()}"
    print(f"\t- {c}:\n\t\t- Type: {dtype}\n\t\t- Unique Values: {uniqueCount}\n\t\t- {detail}")

Column Names:
	- GLOBAL UNIQUE IDENTIFIER:
		- Type: object
		- Unique Values: 10311822
		- Max Length: 47
	- LAST EDITED DATE:
		- Type: datetime64[ns]
		- Unique Values: 604791
		- Range: 2020-02-26 08:38:23, 2026-10-11 11:43:14.118760
	- TAXONOMIC ORDER:
		- Type: int64
		- Unique Values: 949
		- Range: 38, 35853
	- CATEGORY:
		- Type: object
		- Unique Values: 8
		- Max Length: 10
	- TAXON CONCEPT ID:
		- Type: object
		- Unique Values: 949
		- Max Length: 16
	- COMMON NAME:
		- Type: object
		- Unique Values: 778
		- Max Length: 46
	- SCIENTIFIC NAME:
		- Type: object
		- Unique Values: 778
		- Max Length: 47
	- SUBSPECIES COMMON NAME:
		- Type: object
		- Unique Values: 172
		- Max Length: 48
	- SUBSPECIES SCIENTIFIC NAME:
		- Type: object
		- Unique Values: 172
		- Max Length: 48
	- EXOTIC CODE:
		- Type: object
		- Unique Values: 3
		- Max Length: 1
	- OBSERVATION COUNT:
		- Type: float64
		- Unique Values: 1827
		- Range: 1.0, 1000000.0
	- BREEDING CODE:
		- Type: object
		- U

[Visualize](https://mermaid.live/edit#pako:eNqlWG1v4kYQ_ivIn3pSiHiJA_hb7g7aKG3vFKGqOiFZiz0BN7bXXa8pOZL_fuNXdu0dE135RJhn531nns3J8rgPlmOB-BywnWDRRmziAX48JmHHxcvgVP-Sf-7_XC9_XT42UjfwB18fVMRfd4-ffrt7_GVsfxjELIJa9lZ_SRPwAkjNeishrdYefRh4PIp47Kra25AUtcQyeAo81-iEZEce6y7Uxyd4vBC7Ho89SGTXmdrbAsYjNMKFD-JSolYPF0JekSGn2bbGvSf6M_pSIuDIZeCZM4EVLMVu3iRKDvQ0uiYdTYqArqUx00QOdFdWXVe2AsAP4p37fxsXTXV0kdbIxJ1V6Km73BuNDY9nsWzHoZio5CYLzSUcNbDuPZRo3qw8P1aI-3S3Pehp3lyVObwX2oFSfik6xU2zA9NGVceDYMvM5vMzKCTbPuQeCwN0Tr4kQPQ9Snp7TNNBWqDT05ynJ3AriXR-mlhXxD1V7PUFa2hivk1BHEAQgWAiagQdyM1IQXHhBT5iCUtMBjhPfrYu08bSWU_HUCK45B4PyYtZA_rufjdlbT8aLca5jdJ_wJPmICsh6i9nP4iU7JHSUgHPDaUq6uP9esASFB-gm--URUmYTzg4oJWeQaLhFJ962lZt7tXDezqHXBt6Mcwr2JQvHbn6_cvdehBiV8jMB4OExztC5GeibKYoiDOpJ7hEwNMTF9L1AxxmuAfd54jEMAHM3TNTEHEWbfP7USWlW8gwdGtWICBBbeeitvO_EzxLlHSouPXy7_XA24P3HKLDBR1BWNp3H4mbkhsK-ZaFbhYH_2bwvt6g-8lcXGm4Z5_v1sv1_R9LrCiGgIsac-H6ypZSFagDoZin71r5ZENuYc8OARcFzOiUahBbQsh2KfcsdSO0xdoCAYcA_tPrWtRLJY_GarEdMTBR0L9izCQfjgTBh-NPvBlyJ0iVvW8FssHMzVJFSzD10nezsLx-muedp9Tr63D4eio5r6PSv_bbSAeeXwj680VFVSTc6RDqFtE3nlGotZHZV4eUvnTKa0WT7-pILTBG25Bok37tNqldWoGrlnCqkqmN10aUdTPNpTaS6pY2Ja_OFeTT0Wiwzq4rXMnCHIWstkhwhavXnqPyNpWtdnA1czMz0w68Yh0dllnHo41WR13DHT5HHFF2MknMqqP1Zu641RAswoa20tuEiD7TXvEEkTE1I7lzrCtrJwLfcqTI4MqKQEQs_9M65bo3ltwDjjPLwa8C_Ow49Jl4HqLvXGysTfyG5xMWf-M8qlXg5t3tLeeJhSn-lSX5Vqr-NdNAIPZBfMpbxHLGs2mhw3JO1tFyhpPx7fViPhvdjGbziX07GqH4BX-fj26uR_bEnt9ObXu2mN-8XVnfC7uT68XidjGxx2PbntvTyeztByf6qok)

In [40]:
from mermaid import Mermaid

Mermaid("""
erDiagram
    category {
        INTEGER category_id PK
        VARCHAR(15) name
    }
    species {
        INTEGER species_id PK
        VARCHAR(50) common_name
        VARCHAR(50) scientific_name
    }
    taxon {
        VARCHAR(20) taxon_concept_id PK
        INTEGER taxonomic_order
        INTEGER category_id FK
        INTEGER species_id FK
        VARCHAR(50) subspecies_common_name
        VARCHAR(50) subspecies_scientific_name
    }
    exotic {
        VARCHAR(5) exotic_code PK
    }
    taxon_exotic {
        INTEGER te_id PK
        VARCHAR(20) taxon_concept_id FK
        VARCHAR(5) exotic_code FK
    }
    breeding_category {
        INTEGER category_id PK
        VARCHAR(5) breeding_category
    }
    breeding {
        VARCHAR(5) breeding_code PK
        INTEGER category_id FK
    }
    country {
        VARCHAR(5) country_code PK
        VARCHAR(10) country
    }
    state {
        VARCHAR(10) state_code PK
        VARCHAR(5) country_code FK
        VARCHAR(50) state
    }
    county {
        VARCHAR(10) county_code PK
        VARCHAR(10) state_code FK
        VARCHAR(30) county
    }
    iba {
        VARCHAR(30) iba_code PK
    }
    locality_type {
        INTEGER type_id PK
        VARCHAR(5) locality_type
    }
    locality {
        VARCHAR(10) locality_id PK
        VARCHAR(10) county_code FK
        VARCHAR(30) iba_code FK
        VARCHAR(200) locality
        INTEGER type_id FK
    }
    observer {
        VARCHAR(15) observer_id PK
        VARCHAR(40) observer_orcid_id
    }
    observation_type {
        INTEGER type_id PK
        VARCHAR(30) observation_type
    }
    protocol {
        VARCHAR(5) protocol_code PK
        INTEGER type_id FK
        VARCHAR(30) protocol_name
    }
    project {
        INTEGER project_identifiers PK
        VARCHAR(100) project_names
        BIT approved
    }
    sampling_event {
        VARCHAR(10) sampling_event_identifier PK
        VARCHAR(10) locality_id FK
        VARCHAR(15) observer_id FK
        VARCHAR(5) protocol_code FK
        INTEGER project_identifiers FK
        FLOAT latitude
        FLOAT longitude
        FLOAT duration_minutes
        FLOAT effort_distance_km
        FLOAT effort_area_ha
        INTEGER number_observers
        BIT all_species_reported
        VARCHAR(10) group_identifier
        TEXT checklist_comments
    }
    observation {
        VARCHAR(50) global_unique_identifier PK
        VARCHAR(10) sampling_event_identifier FK
        INTEGER te_id FK
        DATETIME last_edited_date
        INTEGER observation_count
        VARCHAR(5) breeding_code FK
        VARCHAR(5) behavior_code
        DATETIME observation_start
        BIT has_media
        BIT reviewed
        TEXT species_comments
    }
    age {
        INTEGER age_id PK
        VARCHAR(10) name
    }
    sex {
        INTEGER sex_id PK
        VARCHAR(15) name
    }
    age_sex {
        INTEGER id PK
        VARCHAR(50) global_unique_identifier FK
        INTEGER age_id FK
        INTEGER sex_id FK
        INTEGER number
    }

    category ||--|{ taxon: category_id
    species ||--|{ taxon: species_id
    taxon ||--|{ taxon_exotic: taxon_concept_id
    exotic ||--|{ taxon_exotic: exotic_code
    taxon_exotic ||--|{ observation: te_id
    breeding_category ||--|{ breeding: category_id
    breeding ||--|{ observation: breeding_code
    age ||--|{ age_sex: age_id
    sex ||--|{ age_sex: sex_id
    observation ||--|{ age_sex: global_unique_identifier
    country ||--|{ state: country_code
    state ||--|{ county: state_code
    county ||--|{ locality: county_code
    iba ||--|{ locality: iba_code
    locality_type ||--|{ locality: type_id
    locality ||--|{ sampling_event: locality_id
    observer ||--|{ sampling_event: observer_id
    observation_type ||--|{ protocol: type_id
    protocol ||--|{ sampling_event: protocol_code
    project ||--|{ sampling_event: project_identifiers
    sampling_event ||--|{ observation: sampling_event_identifier
""")

In [41]:
category = df1[[
    "CATEGORY"
]].drop_duplicates().dropna().reset_index(drop = True)
category.insert(0, "category_id", category.index + 1)
category.rename(columns = {
    "CATEGORY": "name"
}, inplace = True)
species = df1[[
    "COMMON NAME",
    "SCIENTIFIC NAME"
]].drop_duplicates().dropna(how = "all").reset_index(drop = True)
species.insert(0, "species_id", species.index + 1)
species.rename(columns = {
    "COMMON NAME": "common_name",
    "SCIENTIFIC NAME": "scientific_name"
}, inplace = True)
taxon = df1[[
    "TAXON CONCEPT ID",
    "TAXONOMIC ORDER",
    "CATEGORY",
    "COMMON NAME",
    "SCIENTIFIC NAME",
    "SUBSPECIES COMMON NAME",
    "SUBSPECIES SCIENTIFIC NAME"
]].drop_duplicates()
taxon = taxon.merge(category, left_on = "CATEGORY", right_on = "name", how = "left")
taxon = taxon.merge(species, left_on = [
    "COMMON NAME",
    "SCIENTIFIC NAME"
], right_on = [
    "common_name",
    "scientific_name"
], how = "left")
taxon.rename(columns = {
    "TAXON CONCEPT ID": "taxon_concept_id",
    "TAXONOMIC ORDER": "taxonomic_order",
    "SUBSPECIES COMMON NAME": "subspecies_common_name",
    "SUBSPECIES SCIENTIFIC NAME": "subspecies_scientific_name"
}, inplace = True)
taxon = taxon[[
    "taxon_concept_id",
    "taxonomic_order",
    "category_id",
    "species_id",
    "subspecies_common_name",
    "subspecies_scientific_name"
]].dropna(subset = [
    "taxon_concept_id"
])
exotic = df1[[
    "EXOTIC CODE"
]].drop_duplicates().dropna()
exotic.rename(columns = {
    "EXOTIC CODE": "exotic_code"
}, inplace = True)
taxon_exotic = df1[[
    "TAXON CONCEPT ID",
    "EXOTIC CODE"
]].drop_duplicates().dropna(subset = [
    "TAXON CONCEPT ID"
])
taxon_exotic.rename(columns = {
    "TAXON CONCEPT ID": "taxon_concept_id",
    "EXOTIC CODE": "exotic_code"
}, inplace = True)
taxon_exotic.insert(0, "te_id", taxon_exotic.index + 1)
breeding_category = df1[[
    "BREEDING CATEGORY"
]].drop_duplicates().dropna().reset_index(drop = True)
breeding_category.insert(0, "category_id", breeding_category.index + 1)
breeding_category.rename(columns = {
    "BREEDING CATEGORY": "breeding_category"
}, inplace = True)
breeding = df1[[
    "BREEDING CODE",
    "BREEDING CATEGORY"
]].drop_duplicates().dropna(subset = [
    "BREEDING CODE"
])
breeding = breeding.merge(breeding_category, left_on = "BREEDING CATEGORY", right_on = "breeding_category", how = "left")
breeding.rename(columns = {
    "BREEDING CODE": "breeding_code"
}, inplace = True)
breeding = breeding[[
    "breeding_code",
    "category_id"
]]
country = df1[[
    "COUNTRY CODE",
    "COUNTRY"
]].drop_duplicates().dropna(subset = [
    "COUNTRY CODE"
])
country.rename(columns = {
    "COUNTRY CODE": "country_code", "COUNTRY": "country"
}, inplace = True)
state = df1[[
    "STATE CODE",
    "COUNTRY CODE",
    "STATE"
]].drop_duplicates().dropna(subset = [
    "STATE CODE"
])
state.rename(columns = {
    "STATE CODE": "state_code",
    "COUNTRY CODE": "country_code",
    "STATE": "state"}, inplace = True)
county = df1[[
    "COUNTY CODE",
    "STATE CODE",
    "COUNTY"
]].drop_duplicates().dropna(subset = [
    "COUNTY CODE"
])
county.rename(columns = {
    "COUNTY CODE": "county_code",
    "STATE CODE": "state_code",
    "COUNTY": "county"
}, inplace = True)
iba = df1[[
    "IBA CODE"
]].drop_duplicates().dropna()
iba.rename(columns = {
    "IBA CODE": "iba_code"
}, inplace = True)
locality_type = df1[[
    "LOCALITY TYPE"
]].drop_duplicates().dropna().reset_index(drop = True)
locality_type.insert(0, "type_id", locality_type.index + 1)
locality_type.rename(columns = {
    "LOCALITY TYPE": "locality_type"
}, inplace = True)
locality = df1[[
    "LOCALITY ID",
    "COUNTY CODE",
    "IBA CODE",
    "LOCALITY",
    "LOCALITY TYPE"
]].drop_duplicates().dropna(subset = [
    "LOCALITY ID"
])
locality = locality.merge(locality_type, left_on = "LOCALITY TYPE", right_on = "locality_type", how = "left")
locality.rename(columns = {
    "LOCALITY ID": "locality_id",
    "COUNTY CODE": "county_code",
    "IBA CODE": "iba_code",
    "LOCALITY": "locality"
}, inplace = True)
locality = locality[[
    "locality_id",
    "county_code",
    "iba_code",
    "locality",
    "type_id"
]]
observer = df1[[
    "OBSERVER ID",
    "OBSERVER ORCID ID"
]].drop_duplicates().dropna(subset = [
    "OBSERVER ID"
])
observer.rename(columns = {
    "OBSERVER ID": "observer_id",
    "OBSERVER ORCID ID": "observer_orcid_id"
}, inplace = True)
observation_type = df1[[
    "OBSERVATION TYPE"
]].drop_duplicates().dropna().reset_index(drop = True)
observation_type.insert(0, "type_id", observation_type.index + 1)
observation_type.rename(columns = {
    "OBSERVATION TYPE": "observation_type"
}, inplace = True)
protocol = df1[[
    "PROTOCOL CODE",
    "OBSERVATION TYPE",
    "PROTOCOL NAME"
]].drop_duplicates().dropna(subset = [
    "PROTOCOL CODE"
])
protocol = protocol.merge(observation_type, left_on = "OBSERVATION TYPE", right_on = "observation_type", how = "left")
protocol.rename(columns = {
    "PROTOCOL CODE": "protocol_code",
    "PROTOCOL NAME": "protocol_name"
}, inplace = True)
protocol = protocol[[
    "protocol_code",
    "type_id",
    "protocol_name"
]]
project = df1[[
    "PROJECT IDENTIFIERS",
    "PROJECT NAMES",
    "APPROVED"
]].drop_duplicates().dropna(subset = [
    "PROJECT IDENTIFIERS"
])
project.rename(columns = {
    "PROJECT IDENTIFIERS": "project_identifiers", 
    "PROJECT NAMES": "project_names",
    "APPROVED": "approved"
}, inplace = True)
project["project_identifiers"] = project["project_identifiers"].astype(int)
sampling_event = df1[[
    "SAMPLING EVENT IDENTIFIER",
    "LOCALITY ID",
    "OBSERVER ID",
    "PROTOCOL CODE", 
    "PROJECT IDENTIFIERS",
    "LATITUDE",
    "LONGITUDE",
    "DURATION MINUTES", 
    "EFFORT DISTANCE KM",
    "EFFORT AREA HA",
    "NUMBER OBSERVERS", 
    "ALL SPECIES REPORTED",
    "GROUP IDENTIFIER",
    "CHECKLIST COMMENTS"
]].drop_duplicates().dropna(subset = [
    "SAMPLING EVENT IDENTIFIER"
])
sampling_event.rename(columns = {
    "SAMPLING EVENT IDENTIFIER": "sampling_event_identifier",
    "LOCALITY ID": "locality_id",
    "OBSERVER ID": "observer_id",
    "PROTOCOL CODE": "protocol_code",
    "PROJECT IDENTIFIERS": "project_identifiers",
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude",
    "DURATION MINUTES": "duration_minutes",
    "EFFORT DISTANCE KM": "effort_distance_km",
    "EFFORT AREA HA": "effort_area_ha",
    "NUMBER OBSERVERS": "number_observers",
    "ALL SPECIES REPORTED": "all_species_reported",
    "GROUP IDENTIFIER": "group_identifier",
    "CHECKLIST COMMENTS": "checklist_comments"
}, inplace = True)
observation = df1[[
    "GLOBAL UNIQUE IDENTIFIER",
    "SAMPLING EVENT IDENTIFIER",
    "TAXON CONCEPT ID",
    "EXOTIC CODE",
    "LAST EDITED DATE",
    "OBSERVATION COUNT",
    "BREEDING CODE",
    "BEHAVIOR CODE",
    "TIME OBSERVATIONS STARTED",
    "HAS MEDIA",
    "REVIEWED",
    "SPECIES COMMENTS"
]].drop_duplicates().dropna(subset = [
    "GLOBAL UNIQUE IDENTIFIER"
])
observation = observation.merge(taxon_exotic, left_on = [
    "TAXON CONCEPT ID",
    "EXOTIC CODE"
], right_on = [
    "taxon_concept_id",
    "exotic_code"
], how = "left")
observation.rename(columns = {
    "GLOBAL UNIQUE IDENTIFIER": "global_unique_identifier",
    "SAMPLING EVENT IDENTIFIER": "sampling_event_identifier",
    "LAST EDITED DATE": "last_edited_date",
    "OBSERVATION COUNT": "observation_count",
    "BREEDING CODE": "breeding_code",
    "BEHAVIOR CODE": "behavior_code",
    "TIME OBSERVATIONS STARTED": "observation_start",
    "HAS MEDIA": "has_media",
    "REVIEWED": "reviewed",
    "SPECIES COMMENTS": "species_comments"
}, inplace = True)
observation = observation[[
    "global_unique_identifier",
    "sampling_event_identifier",
    "te_id",
    "last_edited_date",
    "observation_count",
    "breeding_code",
    "behavior_code",
    "observation_start",
    "has_media",
    "reviewed",
    "species_comments"
]]
df_agesex_exploded = df1[[
    "GLOBAL UNIQUE IDENTIFIER",
    "AGE/SEX"
]].explode("AGE/SEX").dropna(subset = [
    "AGE/SEX"
])
df_agesex_exploded["AGE"] = df_agesex_exploded["AGE/SEX"].apply(lambda x: x.get("AGE"))
df_agesex_exploded["SEX"] = df_agesex_exploded["AGE/SEX"].apply(lambda x: x.get("SEX"))
df_agesex_exploded["number"] = df_agesex_exploded["AGE/SEX"].apply(lambda x: x.get("QUANTITY"))
age = df_agesex_exploded[["AGE"]].drop_duplicates().dropna().reset_index(drop = True)
age.insert(0, "age_id", age.index + 1)
age.rename(columns={"AGE": "name"}, inplace = True)
sex = df_agesex_exploded[["SEX"]].drop_duplicates().dropna().reset_index(drop = True)
sex.insert(0, "sex_id", sex.index + 1)
sex.rename(columns={"SEX": "name"}, inplace = True)
age_sex = df_agesex_exploded.merge(age, left_on = "AGE", right_on = "name", how = "left")
age_sex = age_sex.merge(sex, left_on = "SEX", right_on = "name", how = "left", suffixes = ("_age", "_sex"))
age_sex.rename(columns = {
    "GLOBAL UNIQUE IDENTIFIER": "global_unique_identifier"
}, inplace = True)
age_sex = age_sex[[
    "global_unique_identifier",
    "age_id",
    "sex_id",
    "number"
]]
age_sex.insert(0, "id", age_sex.index + 1)
print("All Database-like Data Frames Created!")

All Database-like Data Frames Created!


In [42]:
os.makedirs("db_like_ebird_data", exist_ok = True)
category.to_parquet("db_like_ebird_data/category.parquet", index = False)
species.to_parquet("db_like_ebird_data/species.parquet", index = False)
exotic.to_parquet("db_like_ebird_data/exotic.parquet", index = False)
taxon.to_parquet("db_like_ebird_data/taxon.parquet", index = False)
taxon_exotic.to_parquet("db_like_ebird_data/taxon_exotic.parquet", index = False)
breeding_category.to_parquet("db_like_ebird_data/breeding_category.parquet", index = False)
breeding.to_parquet("db_like_ebird_data/breeding.parquet", index = False)
country.to_parquet("db_like_ebird_data/country.parquet", index = False)
state.to_parquet("db_like_ebird_data/state.parquet", index = False)
county.to_parquet("db_like_ebird_data/county.parquet", index = False)
iba.to_parquet("db_like_ebird_data/iba.parquet", index = False)
locality_type.to_parquet("db_like_ebird_data/locality_type.parquet", index = False)
locality.to_parquet("db_like_ebird_data/locality.parquet", index = False)
observer.to_parquet("db_like_ebird_data/observer.parquet", index = False)
observation_type.to_parquet("db_like_ebird_data/observation_type.parquet", index = False)
protocol.to_parquet("db_like_ebird_data/protocol.parquet", index = False)
project.to_parquet("db_like_ebird_data/project.parquet", index = False)
sampling_event.to_parquet("db_like_ebird_data/sampling_event.parquet", index = False)
observation.to_parquet("db_like_ebird_data/observation.parquet", index = False)
age.to_parquet("db_like_ebird_data/age.parquet", index = False)
sex.to_parquet("db_like_ebird_data/sex.parquet", index = False)
age_sex.to_parquet("db_like_ebird_data/age_sex.parquet", index = False)
print("All Database-like Parquet Files Created!")

All Database-like Parquet Files Created!


In [7]:
gdf = gpd.GeoDataFrame(df1, geometry = gpd.points_from_xy(df1["LONGITUDE"], df1["LATITUDE"]), crs = "EPSG:4326")
gdf.to_file("chile_ebd_relSep-2026.shp", driver = "ESRI Shapefile")

C:\Users\lucia\AppData\Local\Temp\ipykernel_27472\3908764371.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("chile_ebd_relSep-2026.shp", driver = "ESRI Shapefile")
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'GLOBAL UNIQUE IDENTIFIER' to 'GLOBAL UNI'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LAST EDITED DATE' to 'LAST EDITE'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field LAST EDITE create as date field, though DateTime requested.
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'TAXONOMIC ORDER' to 'TAXONOMIC'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'TAXON 